# ✈️ MBA em Redes Neurais Aplicadas a Negócios
## Precificação Dinâmica e Yield Management em e-Commerce de Companhias Aéreas

**Versão revisada conforme a avaliação do professor**

### Objetivo
Este notebook mantém a arquitetura **neuro-simbólica** do projeto original, mas incorpora as correções solicitadas na revisão:

- comparação empírica entre **Regressão Logística (baseline)** e **MLP** no mesmo split;
- discussão e teste de uma alternativa de **regras determinísticas**;
- avaliação de robustez com **múltiplas seeds**;
- conexão entre probabilidade prevista e **receita esperada em R$**;
- análise econômica com **CAPEX, OPEX, ROI e payback**, explicitando que os valores são premissas de cenário;
- critérios objetivos de **Data Drift e Concept Drift**;
- seção explícita de **limitações**;
- separação entre resultado técnico do experimento sintético e projeção econômica de negócio.

> **Importante:** os dados usados neste PoC são sintéticos. Portanto, os resultados de ML demonstram a viabilidade do pipeline e não comprovam, por si só, um ganho de receita em produção.

## 1. Contexto do problema e hipótese de negócio

No setor aéreo, o assento é um ativo perecível. A empresa precisa equilibrar o risco de **spoilage** (assento vazio na decolagem) e **spill** (venda antecipada por preço baixo, impedindo a captura de disposição a pagar maior).

A hipótese deste trabalho é:

> Uma MLP pode estimar melhor a probabilidade de compra em diferentes combinações de preço e contexto do que um modelo linear simples, permitindo ao agente prescritivo selecionar uma tarifa que maximize a receita esperada, respeitando guardrails de negócio.

### Métricas

**Técnicas:** ROC-AUC, acurácia e Log Loss.

**Negócio:** receita esperada por assento e receita incremental simulada.

**Hipótese econômica:** a projeção de **+4,5% no RASK** e **R$ 45.000/mês por rota** é tratada como **premissa de cenário para ROI**, e não como consequência direta do AUC da rede neural.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    classification_report,
    confusion_matrix,
    roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)

## 2. Geração do dataset sintético

O dataset reproduz o desenho do trabalho original: 4.000 buscas de passagens com seis variáveis preditoras.

A variável `venda_realizada` é gerada por uma função logística conhecida. Isso é útil para uma PoC didática, mas também constitui uma limitação importante: o modelo está sendo avaliado em dados artificiais cuja estrutura geradora foi definida pelo próprio autor.

In [ ]:
def gerar_dataset_passagens_aereas(n_amostras=4000, seed=42):
    rng = np.random.default_rng(seed)

    dias_ate_decolagem = rng.integers(1, 91, size=n_amostras)
    taxa_ocupacao_atual = rng.uniform(0.10, 0.95, size=n_amostras)
    preco_concorrente = rng.uniform(300, 1500, size=n_amostras)
    historico_buscas_24h = rng.integers(50, 1001, size=n_amostras)
    dia_da_semana = rng.integers(0, 7, size=n_amostras)

    ruido_preco = rng.normal(0, 120, size=n_amostras)
    fator_urgencia = np.where(
        dias_ate_decolagem < 7, 1.35,
        np.where(dias_ate_decolagem < 21, 1.12, 0.92)
    )

    preco_ofertado = np.clip(
        preco_concorrente * fator_urgencia + ruido_preco,
        250, 1800
    )

    razao_preco = preco_ofertado / preco_concorrente
    fator_proximidade = 1 / (np.log1p(dias_ate_decolagem) + 0.3)
    fator_demanda = (historico_buscas_24h - 50) / 950
    fator_dia_nobre = np.isin(dia_da_semana, [3, 4, 6]).astype(float) * 0.40

    logit = (
        3.0
        - 4.20 * razao_preco
        + 2.30 * fator_proximidade
        + 1.10 * fator_demanda
        + fator_dia_nobre
        - 0.60 * (taxa_ocupacao_atual - 0.50)
        + rng.normal(0, 0.40, size=n_amostras)
    )

    probabilidade_compra = 1 / (1 + np.exp(-logit))
    venda_realizada = (
        rng.uniform(0, 1, size=n_amostras) < probabilidade_compra
    ).astype(int)

    return pd.DataFrame({
        "dias_ate_decolagem": dias_ate_decolagem,
        "taxa_ocupacao_atual": np.round(taxa_ocupacao_atual, 3),
        "preco_concorrente": np.round(preco_concorrente, 2),
        "historico_buscas_24h": historico_buscas_24h,
        "dia_da_semana": dia_da_semana,
        "preco_ofertado": np.round(preco_ofertado, 2),
        "venda_realizada": venda_realizada
    })

df_voos = gerar_dataset_passagens_aereas(4000, SEED)

display(df_voos.head())
print("Dimensão:", df_voos.shape)
print("Taxa de conversão:", f"{df_voos['venda_realizada'].mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df_voos,
    x="preco_ofertado",
    hue="venda_realizada",
    bins=30,
    kde=True,
    ax=axes[0]
)
axes[0].set_title("Distribuição do preço ofertado por resultado")

sns.boxplot(
    data=df_voos,
    x="venda_realizada",
    y="taxa_ocupacao_atual",
    ax=axes[1]
)
axes[1].set_title("Ocupação atual por resultado da venda")

plt.tight_layout()
plt.show()

## 3. Split e pré-processamento

O split é realizado **antes** do ajuste do `StandardScaler`, evitando vazamento de informação do conjunto de teste.

Todos os modelos abaixo utilizarão exatamente o mesmo `X_train`, `X_test`, `y_train` e `y_test`.

In [ ]:
FEATURES = [
    "dias_ate_decolagem",
    "taxa_ocupacao_atual",
    "preco_concorrente",
    "historico_buscas_24h",
    "dia_da_semana",
    "preco_ofertado"
]
TARGET = "venda_realizada"

X = df_voos[FEATURES].copy()
y = df_voos[TARGET].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Conversão treino:", f"{y_train.mean():.2%}")
print("Conversão teste :", f"{y_test.mean():.2%}")

## 4. Baseline: Regressão Logística

Antes de justificar uma rede neural, precisamos verificar se um modelo simples já resolve o problema.

A Regressão Logística é barata, rápida e interpretável. Se seu desempenho fosse suficiente para o objetivo de negócio, a MLP poderia representar complexidade desnecessária.

A comparação será feita no **mesmo split** e com as mesmas variáveis.

In [ ]:
def avaliar_probabilidades(y_true, probas, threshold=0.5):
    pred = (probas >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, probas),
        "Log Loss": log_loss(y_true, probas)
    }

baseline = LogisticRegression(
    max_iter=2000,
    random_state=SEED
)
baseline.fit(X_train, y_train)

proba_baseline = baseline.predict_proba(X_test)[:, 1]
metricas_baseline = avaliar_probabilidades(y_test, proba_baseline)

pd.DataFrame([metricas_baseline], index=["Regressão Logística"])

## 5. Rede Neural MLP

A MLP mantém a arquitetura do projeto original, com camadas densas, ReLU, regularização e Dropout.

A justificativa para testar a MLP é sua capacidade de representar relações não lineares entre preço, ocupação, urgência e demanda, que podem ser úteis para construir uma curva de receita esperada mais flexível.

In [ ]:
def construir_mlp(input_dim, seed=42):
    tf.keras.utils.set_random_seed(seed)

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=keras.regularizers.l2(1e-4)
        ),
        layers.Dropout(0.20),
        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=keras.regularizers.l2(1e-4)
        ),
        layers.Dropout(0.20),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model = construir_mlp(X_train.shape[1], SEED)

early_stop = callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

proba_mlp = model.predict(X_test, verbose=0).ravel()
metricas_mlp = avaliar_probabilidades(y_test, proba_mlp)

pd.DataFrame(
    [metricas_baseline, metricas_mlp],
    index=["Regressão Logística", "MLP"]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Treino")
axes[0].plot(history.history["val_loss"], label="Validação")
axes[0].set_title("Loss por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Binary Cross Entropy")
axes[0].legend()

axes[1].plot(history.history["auc"], label="Treino")
axes[1].plot(history.history["val_auc"], label="Validação")
axes[1].set_title("AUC por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Comparação empírica: ML tradicional vs. Redes Neurais

A decisão pela MLP deve ser baseada no ganho observado, e não apenas em sua maior complexidade.

O ponto de decisão é:

> **A MLP melhora as métricas de forma consistente o suficiente para justificar o custo adicional de desenvolvimento e sustentação?**

In [ ]:
comparacao = pd.DataFrame(
    [metricas_baseline, metricas_mlp],
    index=["Regressão Logística", "MLP"]
)

comparacao["Δ AUC vs. baseline"] = (
    comparacao["ROC-AUC"] - comparacao.loc["Regressão Logística", "ROC-AUC"]
)
comparacao["Δ Accuracy vs. baseline"] = (
    comparacao["Accuracy"] - comparacao.loc["Regressão Logística", "Accuracy"]
)

display(comparacao)

fpr_b, tpr_b, _ = roc_curve(y_test, proba_baseline)
fpr_m, tpr_m, _ = roc_curve(y_test, proba_mlp)

plt.figure(figsize=(8, 6))
plt.plot(fpr_b, tpr_b, label=f"Logística (AUC={metricas_baseline['ROC-AUC']:.3f})")
plt.plot(fpr_m, tpr_m, label=f"MLP (AUC={metricas_mlp['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — mesmo conjunto de teste")
plt.legend()
plt.show()

## 7. Alternativa de automação: regra determinística

Uma solução de IA não deve ser escolhida automaticamente quando uma regra simples poderia resolver o problema.

Como alternativa, construímos uma política de preço baseada em faixas de ocupação, proximidade do voo e demanda.

Essa política é deliberadamente simples. Seu objetivo é funcionar como referência de **automação sem ML**.

In [ ]:
def preco_regra_deterministica(row):
    preco_base = row["preco_concorrente"]

    if row["taxa_ocupacao_atual"] >= 0.80 and row["dias_ate_decolagem"] <= 7:
        preco = max(850, preco_base * 0.98)
    elif row["taxa_ocupacao_atual"] < 0.40 and row["dias_ate_decolagem"] <= 15:
        preco = min(900, preco_base * 0.90)
    elif row["historico_buscas_24h"] >= 800:
        preco = preco_base * 1.10
    else:
        preco = preco_base

    return float(np.clip(preco, 400, 2000))

df_voos["preco_regra"] = df_voos.apply(preco_regra_deterministica, axis=1)

display(
    df_voos[
        ["taxa_ocupacao_atual", "dias_ate_decolagem",
         "historico_buscas_24h", "preco_concorrente", "preco_regra"]
    ].head(10)
)

### Por que a regra não substitui necessariamente a MLP?

A regra é simples, barata e auditável, mas precisa ser definida manualmente. Ela trabalha com faixas e limiares fixos e não estima diretamente a probabilidade de compra para cada preço.

A MLP, por outro lado, pode produzir uma curva contínua de `P(compra | preço, contexto)`, permitindo ao agente pesquisar o preço que maximiza a receita esperada.

Assim, a IA só se justifica se o ganho econômico e preditivo superar o custo adicional.

## 8. Robustez estatística — múltiplas seeds

A primeira versão do trabalho reportava uma única execução com `seed=42`.

Para reduzir o risco de atribuir a uma única amostra um ganho que pode ser apenas sorte de treinamento, repetimos o experimento com três seeds: **10, 42 e 100**.

O resultado será reportado como **média ± desvio padrão**.

In [ ]:
def executar_experimento(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    X_train_raw_s, X_test_raw_s, y_train_s, y_test_s = train_test_split(
        X, y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    scaler_s = StandardScaler()
    X_train_s = scaler_s.fit_transform(X_train_raw_s)
    X_test_s = scaler_s.transform(X_test_raw_s)

    lr = LogisticRegression(max_iter=2000, random_state=seed)
    lr.fit(X_train_s, y_train_s)
    p_lr = lr.predict_proba(X_test_s)[:, 1]

    nn = construir_mlp(X_train_s.shape[1], seed)

    es = callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=12,
        restore_best_weights=True
    )

    nn.fit(
        X_train_s, y_train_s,
        validation_split=0.20,
        epochs=100,
        batch_size=64,
        callbacks=[es],
        verbose=0
    )

    p_nn = nn.predict(X_test_s, verbose=0).ravel()

    m_lr = avaliar_probabilidades(y_test_s, p_lr)
    m_nn = avaliar_probabilidades(y_test_s, p_nn)

    return {
        "seed": seed,
        "modelo": "Regressão Logística",
        **m_lr
    }, {
        "seed": seed,
        "modelo": "MLP",
        **m_nn
    }

resultados = []
for seed in [10, 42, 100]:
    r_lr, r_nn = executar_experimento(seed)
    resultados.extend([r_lr, r_nn])

df_robustez = pd.DataFrame(resultados)
display(df_robustez)

resumo_robustez = (
    df_robustez
    .groupby("modelo")[["Accuracy", "ROC-AUC", "Log Loss"]]
    .agg(["mean", "std"])
)

display(resumo_robustez)

## 9. Receita esperada e conexão com o impacto de negócio

A métrica técnica precisa ser traduzida para uma métrica de negócio.

Para cada preço candidato:

\[
E[Receita(p)] = p 	imes \hat{P}(Compra \mid p, contexto)
\]

O agente utiliza a MLP como sensor de conversão e escolhe o preço que maximiza a receita esperada dentro dos limites de governança.

In [ ]:
FEATURES_AGENTE = FEATURES

def estimar_receita_por_preco(modelo, scaler, contexto, precos):
    cenarios = []

    for preco in precos:
        cenario = contexto.copy()
        cenario["preco_ofertado"] = preco
        cenarios.append([cenario[c] for c in FEATURES_AGENTE])

    X_cenarios = scaler.transform(pd.DataFrame(cenarios, columns=FEATURES_AGENTE))
    probas = modelo.predict(X_cenarios, verbose=0).ravel()

    resultado = pd.DataFrame({
        "preco": precos,
        "probabilidade_compra": probas
    })
    resultado["receita_esperada"] = (
        resultado["preco"] * resultado["probabilidade_compra"]
    )

    return resultado

contexto_exemplo = {
    "dias_ate_decolagem": 3,
    "taxa_ocupacao_atual": 0.85,
    "preco_concorrente": 1200.0,
    "historico_buscas_24h": 900,
    "dia_da_semana": 4,
    "preco_ofertado": 1200.0
}

precos = np.linspace(400, 2000, 161)
curva_receita = estimar_receita_por_preco(
    model, scaler, contexto_exemplo, precos
)

melhor = curva_receita.loc[curva_receita["receita_esperada"].idxmax()]

print(f"Preço ótimo simulado: R$ {melhor['preco']:,.2f}")
print(f"Probabilidade estimada: {melhor['probabilidade_compra']:.2%}")
print(f"Receita esperada: R$ {melhor['receita_esperada']:,.2f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(curva_receita["preco"], curva_receita["receita_esperada"])
plt.axvline(
    melhor["preco"],
    linestyle="--",
    label=f"Ótimo = R$ {melhor['preco']:,.0f}"
)
plt.xlabel("Preço ofertado (R$)")
plt.ylabel("Receita esperada por assento (R$)")
plt.title("Curva de receita esperada e preço ótimo")
plt.legend()
plt.show()

## 10. Guardrails e agente prescritivo

A otimização estatística não pode operar sem governança.

Neste PoC:

- **piso rígido:** R$ 400;
- **teto rígido:** R$ 2.000;
- escassez: ocupação ≥ 80% e ≤ 7 dias;
- spoilage: ocupação < 40% e ≤ 15 dias;
- surge de demanda: buscas ≥ 800.

O agente primeiro calcula a tarifa ótima e depois aplica os limites de negócio.

In [ ]:
class AgentePrecificacaoAerea:
    def __init__(self, modelo, scaler):
        self.modelo = modelo
        self.scaler = scaler
        self.features = FEATURES_AGENTE

    def recomendar(self, contexto, num_passos=161):
        piso = 400.0
        teto = 2000.0
        regras = []

        if (
            contexto["taxa_ocupacao_atual"] >= 0.80
            and contexto["dias_ate_decolagem"] <= 7
        ):
            piso = max(
                850.0,
                contexto["preco_concorrente"] * 0.98
            )
            regras.append("YIELD-ESCASSEZ")

        if (
            contexto["taxa_ocupacao_atual"] < 0.40
            and contexto["dias_ate_decolagem"] <= 15
        ):
            teto = min(teto, 900.0)
            regras.append("SPOILAGE")

        if contexto["historico_buscas_24h"] >= 800:
            piso = min(teto, piso * 1.25)
            regras.append("DEMAND-SURGE")

        if piso > teto:
            piso = teto

        precos = np.linspace(piso, teto, num_passos)
        curva = estimar_receita_por_preco(
            self.modelo, self.scaler, contexto, precos
        )

        melhor = curva.loc[curva["receita_esperada"].idxmax()]

        return {
            "preco_otimizado_brl": float(melhor["preco"]),
            "probabilidade_conversao": float(melhor["probabilidade_compra"]),
            "receita_esperada_brl": float(melhor["receita_esperada"]),
            "piso_efetivo_brl": float(piso),
            "teto_efetivo_brl": float(teto),
            "regras_acionadas": regras
        }

agente = AgentePrecificacaoAerea(model, scaler)
decisao = agente.recomendar(contexto_exemplo)

decisao

## 11. Impacto econômico: simulação de receita em R$

O professor solicitou que o resultado técnico fosse traduzido para dinheiro.

Para evitar uma conclusão causal indevida, fazemos duas análises distintas:

1. **Impacto preditivo/simulado:** comparar a receita esperada usando a probabilidade prevista pelo modelo.
2. **Cenário financeiro:** usar as premissas de negócio do projeto para estimar ROI.

A segunda análise **não deve ser apresentada como prova de que AUC = X gera automaticamente R$ Y**. Ela é uma projeção para avaliação de viabilidade.

In [ ]:
def receita_media_simulada(modelo, scaler, df_contexto, preco_col="preco_ofertado"):
    cenarios = df_contexto[FEATURES].copy()
    X_s = scaler.transform(cenarios)
    probas = modelo.predict(X_s, verbose=0).ravel()

    receita = df_contexto[preco_col].to_numpy() * probas
    return float(np.mean(receita)), float(np.sum(receita))

receita_mlp_media, receita_mlp_total = receita_media_simulada(
    model, scaler, X_test_raw
)

# Para a regressão logística, o mesmo cálculo usa a mesma amostra de teste.
proba_lr_test = baseline.predict_proba(X_test)[:, 1]
receita_lr_total = float(
    np.sum(X_test_raw["preco_ofertado"].to_numpy() * proba_lr_test)
)

print(f"Receita esperada simulada - MLP:       R$ {receita_mlp_total:,.2f}")
print(f"Receita esperada simulada - Logística: R$ {receita_lr_total:,.2f}")
print(
    "Variação simulada: "
    f"{(receita_mlp_total / receita_lr_total - 1):.2%}"
)

### 11.1 Cenário financeiro para ROI

As seguintes premissas são utilizadas **como cenário ilustrativo do projeto**, e não como resultado observado no dataset sintético:

| Item | Premissa |
|---|---:|
| CAPEX de construção | R$ 35.000 |
| OPEX anual | R$ 30.000 |
| Custo total no Ano 1 | R$ 65.000 |
| Receita incremental projetada | R$ 45.000/mês |
| Benefício anual projetado | R$ 540.000 |
| Horizonte | 12 meses |

O benefício de R$ 45.000/mês corresponde à premissa de negócio do projeto para uma malha simulada de 10 rotas. O valor não é inferido diretamente do AUC.

In [ ]:
CAPEX = 35_000.00
OPEX_ANUAL = 30_000.00
BENEFICIO_MENSAL = 45_000.00

CUSTO_ANO_1 = CAPEX + OPEX_ANUAL
BENEFICIO_ANUAL = BENEFICIO_MENSAL * 12

ROI_ANO_1 = (BENEFICIO_ANUAL - CUSTO_ANO_1) / CUSTO_ANO_1
PAYBACK_MESES = CUSTO_ANO_1 / BENEFICIO_MENSAL

roi_df = pd.DataFrame({
    "Indicador": [
        "CAPEX",
        "OPEX anual",
        "Custo total Ano 1",
        "Benefício anual projetado",
        "ROI Ano 1",
        "Payback estimado (meses)"
    ],
    "Valor": [
        CAPEX,
        OPEX_ANUAL,
        CUSTO_ANO_1,
        BENEFICIO_ANUAL,
        ROI_ANO_1,
        PAYBACK_MESES
    ]
})

display(roi_df)

print(f"ROI Ano 1: {ROI_ANO_1:.2%}")
print(f"Payback: {PAYBACK_MESES:.1f} meses")

### Interpretação econômica

O projeto só deve avançar para produção se o benefício real observado em piloto superar o custo incremental da solução.

Por isso, o ROI acima deve ser tratado como **cenário de viabilidade**, sujeito à validação em Shadow Mode e A/B test.

A recomendação é não afirmar que o projeto "gera 730% de ROI" com base no dataset sintético. O correto é dizer que **sob as premissas financeiras adotadas, o cenário apresenta ROI estimado de aproximadamente 730,8% no primeiro ano**.

## 12. MLOps: Data Drift e Concept Drift

O plano original já previa monitoramento e retreinamento semanal. Nesta versão são definidos gatilhos objetivos:

### Data Drift
Monitoramento diário pelo teste de Kolmogorov-Smirnov.

**Ação:** se `KS > 0,15` e `p < 0,05` por mais de 3 dias consecutivos, executar retreinamento emergencial.

### Concept Drift
Monitoramento semanal da diferença entre probabilidade prevista e conversão observada.

**Ação:** se o desvio absoluto ultrapassar **5 pontos percentuais**, retirar temporariamente o modelo de produção, retornar ao baseline de segurança e abrir revisão manual.

Esses limiares são critérios operacionais definidos para o PoC; em produção devem ser calibrados com histórico real.

In [ ]:
from scipy.stats import ks_2samp

def verificar_data_drift(referencia, atual, ks_limite=0.15, p_limite=0.05):
    estatistica, p_valor = ks_2samp(referencia, atual)

    alerta = (
        estatistica > ks_limite
        and p_valor < p_limite
    )

    return {
        "KS": estatistica,
        "p_valor": p_valor,
        "alerta": alerta
    }

def verificar_concept_drift(probabilidade_prevista, conversao_real, limite_pp=0.05):
    desvio = abs(
        np.mean(probabilidade_prevista) - np.mean(conversao_real)
    )

    return {
        "desvio_absoluto": desvio,
        "limite": limite_pp,
        "alerta": desvio > limite_pp
    }

print("Funções de monitoramento definidas.")
print("Data Drift: KS > 0,15 e p < 0,05.")
print("Concept Drift: desvio > 5 p.p.")

## 13. Limitações

1. **Dataset 100% sintético:** as 4.000 observações são geradas por uma função logística definida no próprio projeto.
2. **Ausência de dados históricos reais:** não há validação externa com reservas, cancelamentos ou preços reais.
3. **Variáveis econômicas simplificadas:** câmbio, QAV, sazonalidade, feriados e outras variáveis podem alterar a elasticidade real.
4. **Receita simulada não equivale a receita observada:** a probabilidade prevista pelo modelo permite uma simulação, mas não comprova causalidade.
5. **ROI é uma projeção de cenário:** CAPEX, OPEX e benefício mensal precisam ser substituídos por dados reais antes de uma decisão de investimento.
6. **A comparação entre modelos é experimental:** o ganho da MLP deve ser confirmado em dados reais e em piloto controlado.
7. **Otimização de preço:** o Grid Search maximiza receita esperada por assento, mas não representa toda a complexidade de uma rede aérea, capacidade, conexões e efeitos de inventário.

### Conclusão metodológica

A PoC demonstra um pipeline coerente de **previsão → otimização → governança**, mas a conclusão executiva deve permanecer limitada à viabilidade técnica e à simulação econômica até que dados reais estejam disponíveis.

## 14. Roadmap recomendado

**Fase 1 — Shadow Mode (30 dias)**  
Executar o modelo em paralelo ao sistema atual, sem alterar os preços exibidos.

**Fase 2 — A/B Test**  
Selecionar duas rotas e comparar 50% do tráfego com precificação baseada em IA contra 50% com o modelo estático.

**Fase 3 — Validação econômica**  
Medir RASK, receita incremental, load factor, margem e custo operacional.

**Fase 4 — Produção controlada**  
Expandir gradualmente apenas se os resultados superarem os critérios de sucesso definidos.

**Fase 5 — MLOps**  
Manter retreinamento semanal, monitoramento diário de Data Drift e monitoramento semanal de Concept Drift.

# 15. Conclusão executiva

A revisão do projeto reforça uma conclusão mais defensável:

- a **Regressão Logística** estabelece o baseline simples;
- a **MLP** é justificável somente se seu ganho for consistente nas múltiplas seeds e relevante para a curva de receita;
- a **regra determinística** permanece como alternativa de menor custo e maior previsibilidade;
- o agente prescritivo transforma a previsão de conversão em decisão de preço;
- os guardrails reduzem o risco operacional;
- o ROI apresentado é uma **projeção baseada em premissas**, não um resultado comprovado pelo dataset sintético;
- a validação definitiva depende de **dados reais, Shadow Mode e A/B Test**.

Assim, o projeto evolui de uma PoC puramente técnica para uma proposta de IA com **baseline, robustez estatística, impacto financeiro, governança e critérios objetivos de implantação**.